In [ ]:
# // MARK: 1. 환경 설정 및 MySQL 서버 시작 =====================================
# MySQL 서버 설치 및 서비스 시작
!apt-get update
!apt-get install mysql-server
!service mysql start
!mysql -u root -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY ''; FLUSH PRIVILEGES;"

# 파이썬 접속용 라이브러리 설치
!pip install mysql-connector-python > /dev/null

import mysql.connector
from mysql.connector import Error

# (주의: 서버 설치 및 시작은 시스템 명령어로 선행되어야 합니다)
print("MySQL 서버 연결을 준비합니다...")

# 접속 설정 (Google Colab의 기본 root 계정 기준)
db_config = {
    'host': '127.0.0.1',
    'user': 'root',
    'password': '',
    'autocommit': True
}

def sql_api(connection, query, description=None):
    """SQL 실행 및 결과 출력을 위한 API 래퍼 함수"""
    if description:
        print(f"\n{description}")
        
    cursor = connection.cursor(dictionary=True) # 결과를 딕셔너리 형태로 반환
    try:
        # Multi-statement는 execute에서 지원하지 않으므로 순차 실행 권장
        cursor.execute(query)
        
        # 결과가 있는 경우(SELECT) 표 형식으로 출력
        if cursor.description:
            rows = cursor.fetchall()
            if rows:
                # 헤더 추출 및 출력
                headers = rows[0].keys()
                header_line = " | ".join(headers)
                print(header_line)
                print("-" * len(header_line))
                for row in rows:
                    print(" | ".join(str(val) for val in row.values()))
            else:
                print("결과가 없습니다.")
    except Error as e:
        # 에러가 발생하면 예외를 던져서 상위 try-except에서 잡게 함
        raise e
    finally:
        cursor.close()

try:
    # 서버 연결
    conn = mysql.connector.connect(**db_config)

    # // MARK: 2. 데이터베이스 및 테이블 설계 =====================================
    sql_api(conn, "DROP DATABASE IF EXISTS mart_db;")
    sql_api(conn, "CREATE DATABASE mart_db;")
    conn.database = 'mart_db' # 사용할 DB 선택

    # 유저 테이블
    sql_api(conn, """
    CREATE TABLE users (
        u_id INT AUTO_INCREMENT PRIMARY KEY,
        name VARCHAR(20),
        city VARCHAR(20),
        level VARCHAR(10) DEFAULT 'Silver'
    );
    """)

    # 상품 테이블
    sql_api(conn, """
    CREATE TABLE products (
        p_id INT AUTO_INCREMENT PRIMARY KEY,
        p_name VARCHAR(50),
        category VARCHAR(20),
        price INT,
        stock INT
    );
    """)

    # 주문 내역 테이블
    sql_api(conn, """
    CREATE TABLE orders (
        o_id INT AUTO_INCREMENT PRIMARY KEY,
        u_id INT,
        p_id INT,
        amount INT,
        o_date DATE,
        FOREIGN KEY (u_id) REFERENCES users(u_id),
        FOREIGN KEY (p_id) REFERENCES products(p_id)
    );
    """)

    # // MARK: 3. 다채로운 데이터 삽입 (Bulk Insert) =====================================
    print("\n데이터를 삽입 중입니다...")

    sql_api(conn, """
    INSERT INTO users (name, city, level) VALUES 
    ('이몽룡', '서울', 'Gold'), ('성춘향', '남원', 'VIP'), 
    ('홍길동', '한양', 'Silver'), ('임꺽정', '구월산', 'Gold'),
    ('심청', '백령도', 'Silver');
    """)

    sql_api(conn, """
    INSERT INTO products (p_name, category, price, stock) VALUES 
    ('갤럭시 S24', '가전', 1200000, 15), ('아이폰 15', '가전', 1300000, 10),
    ('불닭볶음면', '식품', 1500, 100), ('삼다수', '식품', 800, 200),
    ('기계식 키보드', '잡화', 150000, 5), ('캠핑 의자', '잡화', 45000, 0);
    """)

    sql_api(conn, """
    INSERT INTO orders (u_id, p_id, amount, o_date) VALUES 
    (1, 1, 1, '2024-05-10'), (2, 3, 5, '2024-05-11'),
    (3, 4, 10, '2024-05-12'), (1, 5, 1, '2024-05-13'),
    (4, 2, 1, '2024-05-14'), (2, 1, 1, '2024-05-15');
    """)

    # // MARK: 4. 실무형 데이터 분석 쿼리 =====================================

    sql_api(conn, """
    SELECT o.o_id AS '번호', u.name AS '고객명', p.p_name AS '상품명', 
           p.price AS '단가', o.amount AS '수량', (p.price * o.amount) AS '총결제액'
    FROM orders o
    JOIN users u ON o.u_id = u.u_id
    JOIN products p ON o.p_id = p.p_id;
    """, "[리포트 1] 전체 주문 상세 내역 (JOIN)")

    sql_api(conn, """
    SELECT category AS '카테고리', SUM(price * stock) AS '재고총액',
           COUNT(*) AS '상품수', ROUND(AVG(price), 0) AS '평균가격'
    FROM products GROUP BY category;
    """, "[리포트 2] 카테고리별 매출 합계 및 평균 단가")

    sql_api(conn, """
    SELECT u.name, u.level, p.p_name, o.o_date
    FROM orders o
    JOIN users u ON o.u_id = u.u_id
    JOIN products p ON o.p_id = p.p_id
    WHERE u.level = 'VIP';
    """, "[리포트 3] VIP 고객의 주문 현황")

    sql_api(conn, "SELECT p_name, stock FROM products WHERE stock < 10;", 
            "[리포트 4] 현재 품절이거나 재고가 10개 미만인 위험 상품")

    # // MARK: 5. 데이터 수정 =====================================
    print("\n[업데이트] '삼다수' 가격 인상 및 '성춘향' 등급 조정")
    sql_api(conn, "UPDATE products SET price = 1000 WHERE p_name = '삼다수';")
    sql_api(conn, "UPDATE users SET level = 'VVIP' WHERE name = '성춘향';")

    sql_api(conn, "SELECT * FROM products WHERE p_name = '삼다수';", "[최종 결과 확인 - 상품]")
    sql_api(conn, "SELECT * FROM users WHERE name = '성춘향';", "[최종 결과 확인 - 유저]")

    # // MARK: 6. 데이터 삭제 및 정리 =====================================
    sql_api(conn, "DELETE FROM products WHERE stock = 0;", "[삭제 시나리오 1] 재고가 0인 상품 제거")
    sql_api(conn, "DELETE FROM users WHERE name = '심청';", "[삭제 시나리오 2] 탈퇴 요청 고객 제거")

    sql_api(conn, "SELECT p_name, stock, price FROM products;", "[최종 확인] 남은 상품 리스트")
    sql_api(conn, "SELECT name, level FROM users;", "[최종 확인] 남은 유저 리스트")

    # // MARK: 7. 고급 분석: 등급별 타겟팅 리포트 =====================================
    sql_api(conn, """
    SELECT name, level, 
           CASE 
               WHEN level = 'VVIP' THEN '전담 매니저 배정 대상'
               WHEN level = 'Gold' THEN '10% 할인 쿠폰 발급'
               ELSE '신규 가입 이벤트 안내'
           END AS '마케팅_액션'
    FROM users;
    """, "[리포트 5] 고객 등급별 구매 권장 문구 (CASE 문)")

    # // MARK: 8. 데이터 무결성 테스트 (에러 케이스) =====================================
    print("\n[테스트] 존재하지 않는 상품(999번)을 주문 시도 (외래키 제약조건 작동 확인)")
    try:
        sql_api(conn, "INSERT INTO orders (u_id, p_id, amount, o_date) VALUES (1, 999, 1, '2024-05-20');")
    except Error:
        print(">> 성공: 외래 키 제약 조건에 의해 잘못된 주문이 차단되었습니다.")

except Error as e:
    print(f"데이터베이스 연결 에러: {e}")

finally:
    if conn.is_connected():
        conn.close()
        print("\nMySQL 연결을 안전하게 종료했습니다.")